# Qwen-Image-2.1 Uncensored (GGUF) — Google Colab / Kaggle Runner

Run **Qwen-Image-2.1 Uncensored GGUF** (`abenzerps/Qwen-Image-2.1-Uncensored-GGUF`) using **ComfyUI** and `ComfyUI-GGUF`.

### Hardware Compatibility:
- **Google Colab:** Free T4 GPU (15GB VRAM) or A100 / L4
- **Kaggle:** Dual or Single T4 GPU

### Components Loaded:
- **Diffusion Transformer:** `qwen-image-2.1-UC-Q4_K_M.gguf` (~4.6 GB in GPU VRAM)
- **Text Encoder:** `qwen3vl_8b_int8_convrot.safetensors` (~9.35 GB in RAM)
- **VAE:** `qwen_image_2.1_vae_bf16.safetensors` (~676 MB)


In [ ]:
# Cell 1: Hardware Check & Package Setup
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq aria2


In [ ]:
# Cell 2: Install ComfyUI and LeeJet ComfyUI-GGUF
import os

%cd /content
if not os.path.exists('ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

%cd /content/ComfyUI
!pip install -q -r requirements.txt

# Install leejet/ComfyUI-GGUF with native Qwen-Image-2.1 GGUF support
%cd /content/ComfyUI/custom_nodes
if not os.path.exists('ComfyUI-GGUF'):
    !git clone https://github.com/leejet/ComfyUI-GGUF.git
%cd /content/ComfyUI/custom_nodes/ComfyUI-GGUF
!pip install -q -r requirements.txt


In [ ]:
# Cell 3: Fast Multi-Threaded Download of Model Weights
import os

diff_dir = '/content/ComfyUI/models/diffusion_models'
clip_dir = '/content/ComfyUI/models/text_encoders'
vae_dir = '/content/ComfyUI/models/vae'

os.makedirs(diff_dir, exist_ok=True)
os.makedirs(clip_dir, exist_ok=True)
os.makedirs(vae_dir, exist_ok=True)

# 1. Diffusion Model (Q4_K_M GGUF - ~4.6 GB)
diff_file = os.path.join(diff_dir, 'qwen-image-2.1-UC-Q4_K_M.gguf')
if not os.path.exists(diff_file):
    print('Downloading Qwen-Image-2.1 UC Q4_K_M GGUF...')
    !aria2c -c -x 16 -s 16 -k 1M "https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF/resolve/main/qwen-image-2.1-UC-Q4_K_M.gguf" -d {diff_dir} -o qwen-image-2.1-UC-Q4_K_M.gguf

# 2. Text Encoder (Int8 ConvRot - ~9.35 GB)
clip_file = os.path.join(clip_dir, 'qwen3vl_8b_int8_convrot.safetensors')
if not os.path.exists(clip_file):
    print('Downloading Qwen3-VL 8B Int8 Text Encoder...')
    !aria2c -c -x 16 -s 16 -k 1M "https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors" -d {clip_dir} -o qwen3vl_8b_int8_convrot.safetensors

# 3. VAE (~676 MB)
vae_file = os.path.join(vae_dir, 'qwen_image_2.1_vae_bf16.safetensors')
if not os.path.exists(vae_file):
    print('Downloading Qwen-Image 2.1 VAE...')
    !aria2c -c -x 16 -s 16 -k 1M "https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors" -d {vae_dir} -o qwen_image_2.1_vae_bf16.safetensors

print('All model components downloaded successfully!')


In [ ]:
# Cell 4: Launch ComfyUI with Cloudflare Public Tunnel
%cd /content/ComfyUI
import os, subprocess, time, re

# Install cloudflared if not present
if not os.path.exists('/content/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared

# Launch ComfyUI in background
comfy_proc = subprocess.Popen([
    'python', 'main.py',
    '--listen', '127.0.0.1',
    '--port', '8188',
    '--lowvram'
])

time.sleep(5)

# Launch Cloudflare tunnel
tunnel_proc = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print('Starting tunnel, waiting for public URL...')
for line in iter(tunnel_proc.stdout.readline, ''):
    if 'trycloudflare.com' in line:
        url_match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if url_match:
            print('\n======================================================')
            print('🚀 COMFYUI IS READY! Click the URL below:')
            print(url_match.group(0))
            print('======================================================\n')
            break
